# Triton Poker: Content Strategy Analysis

**A YouTube content-strategy analysis for Triton Poker Series, built on a Databricks medallion pipeline.**

Triton Poker describes itself as "shaping the future of poker as a content powerhouse, creating
the most trusted and compelling destination for premium poker storytelling, livestreams, and
global fan engagement."

This project asks a straightforward question of the data:

**How well does Triton's content strategy — as reflected in its ~950-video YouTube catalog —
live up to that positioning as a storytelling brand, and what actually drives engagement across
the catalog?**

To answer it, the analysis builds an end-to-end pipeline — ingesting the catalog from the YouTube
Data API, classifying each video along several content dimensions, and examining engagement
patterns — while staying honest about what public data can and cannot show.

*Pipeline: YouTube Data API → bronze (raw) → silver (typed & classified) → gold (curated analytics),
all in Databricks. See the README for architecture, stack, and engineering practices.*

## Setup & Ingestion

*Raw data is ingested from the YouTube Data API v3 into the **bronze layer** — stored as a
**Delta** table (`workspace.youtube.bronze_videos_raw`) in **Unity Catalog**, organized under
a dedicated `youtube` schema alongside the silver and gold layers. Each row preserves the
complete, untransformed API response, providing an immutable raw record that downstream layers
parse and enrich.*

*The API key is read from a **Databricks secret scope** (not hardcoded), and ingestion uses the
channel's uploads playlist rather than the costly `search` endpoint to **minimize API quota cost**
(~40 of 10,000 daily units per full run). The ingestion logic is also available as a standalone,
reusable script (`ingest_youtube.py`) in the repo. This section can be skipped; the analysis
begins at the Silver Layer section.*

### Environment & Connectivity Check

Verifies that the notebook environment can reach the YouTube Data API v3 over the
public internet before any real ingestion runs. Databricks serverless compute must
allow outbound HTTPS for the REST-based ingestion in later cells to work.

A returned HTTP status of any kind (including `404` for the bare base path) confirms
the endpoint is reachable. A timeout or connection error would indicate outbound
network access is blocked, in which case ingestion would need to run elsewhere and
land its output as a file.

In [0]:
import requests
r = requests.get("https://www.googleapis.com/youtube/v3/", timeout=10)
print("Status:", r.status_code)
print("Reachable:", r.status_code in (200, 400, 401, 403, 404))

Status: 404
Reachable: True


### Load API Credentials from Databricks Secrets

Reads the YouTube Data API key from a Databricks secret scope
(`scope="youtube"`, `key="api_key"`) rather than hardcoding it in the notebook.

This keeps the credential out of the notebook source and out of version control —
the key is stored in an encrypted secret scope and is automatically redacted from
notebook output. The check prints only whether the key loaded and its length, never
the value itself.

**Setup (one-time, via Databricks CLI):**
- `databricks secrets create-scope --scope youtube`
- `databricks secrets put --scope youtube --key api_key`

In [0]:
API_KEY = dbutils.secrets.get(scope="youtube", key="api_key")
print("Key loaded:", bool(API_KEY), "| length:", len(API_KEY))

Key loaded: True | length: 39


### Bronze Layer — Raw Ingestion

Ingests the full video catalog for the target channel from the YouTube Data API v3
and lands it as the **bronze** layer: raw, as-received, with no cleaning or typing.

**Ingestion path (cost-optimized):** rather than the expensive `search` endpoint
(100 quota units/call), this resolves the channel's *uploads playlist* and pages
through it (`playlistItems`, 1 unit/page), then batch-fetches video metadata
(`videos`, 1 unit per 50 IDs). A full run of ~948 videos costs ~40 of the 10,000
daily quota units.

**Bronze design:** one row per video. Each row preserves the complete API response
verbatim in `raw_json`, plus two provenance fields — `video_id` (table key) and
`ingested_at` (UTC observation time). No parsing, validation, or deduplication
happens here; all interpretation is deferred to the silver layer. This keeps
ingestion robust (it can't break on unexpected fields) and preserves every field
for downstream use, even ones not yet consumed.

**Output:** `workspace.youtube.bronze_videos_raw` (Delta, overwrite mode).

In [0]:
import requests
import json
from datetime import datetime, timezone
from pyspark.sql import Row

API_KEY = dbutils.secrets.get(scope="youtube", key="api_key")
BASE = "https://www.googleapis.com/youtube/v3"
HANDLE = "TritonPoker"

def get_channel():
    r = requests.get(f"{BASE}/channels", params={
        "part": "snippet,statistics,contentDetails",
        "forHandle": HANDLE, "key": API_KEY})
    r.raise_for_status()
    return r.json()["items"][0]

def get_all_video_ids(uploads_playlist_id):
    ids, token = [], None
    while True:
        params = {"part": "contentDetails", "playlistId": uploads_playlist_id,
                  "maxResults": 50, "key": API_KEY}
        if token:
            params["pageToken"] = token
        r = requests.get(f"{BASE}/playlistItems", params=params)
        r.raise_for_status()
        data = r.json()
        ids += [it["contentDetails"]["videoId"] for it in data["items"]]
        token = data.get("nextPageToken")
        if not token:
            break
    return ids

def get_video_details(video_ids):
    videos = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        r = requests.get(f"{BASE}/videos", params={
            "part": "snippet,statistics,contentDetails,liveStreamingDetails",
            "id": ",".join(batch), "key": API_KEY})
        r.raise_for_status()
        videos += r.json()["items"]
    return videos

# --- run ingestion ---
ingested_at = datetime.now(timezone.utc).isoformat()
channel = get_channel()
uploads = channel["contentDetails"]["relatedPlaylists"]["uploads"]
video_ids = get_all_video_ids(uploads)
videos = get_video_details(video_ids)
print(f"Channel: {channel['snippet']['title']} | videos fetched: {len(videos)}")

# --- write bronze: one row per video, raw JSON preserved + ingest timestamp ---
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.youtube")

rows = [Row(
    video_id=v["id"],
    ingested_at=ingested_at,
    raw_json=json.dumps(v)
) for v in videos]

bronze_df = spark.createDataFrame(rows)
(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.youtube.bronze_videos_raw"))

print(f"Wrote {bronze_df.count()} rows to workspace.youtube.bronze_videos_raw")

Channel: Triton Poker | videos fetched: 948
Wrote 948 rows to workspace.youtube.bronze_videos_raw


## Silver Layer — Cleaned & Classified

Before parsing, a look at the raw structure we're working with — the fields available in
each bronze record, which determine what silver extracts and how it classifies.

In [0]:
import json
rows = spark.table("workspace.youtube.bronze_videos_raw").limit(1).collect()
sample = json.loads(rows[0]["raw_json"])
# show the structure: top-level keys and the keys within each section
for section in ["snippet", "statistics", "contentDetails", "liveStreamingDetails"]:
    if section in sample:
        print(f"{section}: {list(sample[section].keys())}")

snippet: ['publishedAt', 'channelId', 'title', 'description', 'thumbnails', 'channelTitle', 'tags', 'categoryId', 'liveBroadcastContent', 'defaultLanguage', 'localized', 'defaultAudioLanguage']
statistics: ['viewCount', 'likeCount', 'favoriteCount', 'commentCount']
contentDetails: ['duration', 'dimension', 'definition', 'caption', 'licensedContent', 'contentRating', 'projection']
liveStreamingDetails: ['actualStartTime', 'actualEndTime', 'scheduledStartTime']


### Transformation Approach

The silver layer parses the raw JSON into typed columns and classifies each video along four
analytical dimensions:

- **`is_live_broadcast`** — was it a live broadcast? (from the API's `liveStreamingDetails`)
- **`duration_bucket`** — short / medium / long (from parsed video duration)
- **`stop_location`** — which Triton event/location (matched from title and description)
- **`game_format`** — cash / tournament / other (matched from title keywords)

Location and content classification rely on *reference data* rather than logic hardcoded into
the transform — so adding a new stop is a data change, not a code change.

### Reference Data: Stop Locations

A lookup table mapping each Triton stop to a canonical name and a regex of its aliases (e.g.
Montenegro also appears as "Budva" or "Sveti Stefan"). The silver transform joins against this
rather than embedding locations in the SQL — separating business reference data from
transformation logic, and making the stop list extensible by `INSERT`.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.youtube.ref_stops AS
SELECT * FROM VALUES
  ('Montenegro',  'montenegro|budva|sveti stefan'),
  ('Jeju',        'jeju'),
  ('Cyprus',      'cyprus|kyrenia'),
  ('Monte Carlo', 'monte.?carlo|monaco'),
  ('Madrid',      'madrid'),
  ('Vietnam',     'vietnam|hoi.?an|hoiana'),
  ('Macau',       'macau'),
  ('Manila',      'manila'),
  ('Bahamas',     'bahamas'),
  ('London',      'london'),
  ('Rozvadov',    'rozvadov')
AS ref_stops(canonical_name, alias_pattern)

num_affected_rows,num_inserted_rows


### Building the Silver Table

A single transformation parses the raw JSON into typed columns, derives `duration_seconds`
from the ISO-8601 duration, joins against `ref_stops` to resolve location (title first, then
description, best match wins), and derives the duration and game-format buckets. `try_cast`
ensures malformed values become `NULL` rather than failing the job. The result is one clean,
classified row per video, rebuilt idempotently via `CREATE OR REPLACE`.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.youtube.silver_videos AS
WITH base AS (
  SELECT
    video_id,
    ingested_at,
    TRIM(raw_json:snippet.title::string)            AS title,
    raw_json:snippet.description::string            AS description,
    raw_json:snippet.publishedAt::timestamp         AS published_at,
    raw_json:contentDetails.duration::string        AS duration_iso,
    raw_json:statistics.viewCount::long             AS view_count,
    raw_json:statistics.likeCount::long             AS like_count,
    raw_json:statistics.commentCount::long          AS comment_count,
    raw_json:liveStreamingDetails IS NOT NULL       AS is_live_broadcast
  FROM workspace.youtube.bronze_videos_raw
),
enriched AS (
  SELECT *,
    COALESCE(try_cast(regexp_extract(duration_iso, r'(\d+)H', 1) AS INT), 0) * 3600
    + COALESCE(try_cast(regexp_extract(duration_iso, r'(\d+)M', 1) AS INT), 0) * 60
    + COALESCE(try_cast(regexp_extract(duration_iso, r'(\d+)S', 1) AS INT), 0) AS duration_seconds
  FROM base
),
stop_matches AS (
  SELECT e.video_id, r.canonical_name,
         CASE WHEN LOWER(e.title) RLIKE r.alias_pattern THEN 1 ELSE 2 END AS match_rank
  FROM enriched e
  JOIN workspace.youtube.ref_stops r
    ON LOWER(e.title) RLIKE r.alias_pattern
    OR LOWER(COALESCE(e.description, '')) RLIKE r.alias_pattern
),
best_stop AS (
  SELECT video_id, canonical_name AS stop_location
  FROM (SELECT video_id, canonical_name,
               ROW_NUMBER() OVER (PARTITION BY video_id ORDER BY match_rank, canonical_name) AS rn
        FROM stop_matches)
  WHERE rn = 1
)
SELECT
  e.video_id,
  e.ingested_at,
  e.title,
  e.description,
  e.published_at,
  e.duration_iso,
  e.duration_seconds,
  e.view_count,
  e.like_count,
  e.comment_count,

  -- DIMENSION 1: live broadcast (API-reliable)
  e.is_live_broadcast,

  -- DIMENSION 2: duration bucket (objective, data-driven boundaries at 3m / 60m)
  CASE
    WHEN e.duration_seconds < 180  THEN 'short'
    WHEN e.duration_seconds < 3600 THEN 'medium'
    ELSE 'long'
  END AS duration_bucket,

  -- DIMENSION 3: stop location (reference-data matched, ~90% coverage)
  s.stop_location,
  s.stop_location IS NOT NULL AS stop_matched,

  -- DIMENSION 4: game format (keyword-derived where determinable; honest 'other')
  CASE
    WHEN LOWER(e.title) RLIKE 'cash game|cash kings' THEN 'cash'
    WHEN LOWER(e.title) RLIKE r'\$[0-9]|event #|final table|main event| nlh| plo|day [0-9]' THEN 'tournament'
    ELSE 'other'
  END AS game_format

FROM enriched e
LEFT JOIN best_stop s ON e.video_id = s.video_id

num_affected_rows,num_inserted_rows


### Validating the Duration Buckets

Confirms the short / medium / long boundaries (3 min, 60 min) align with natural gaps in the
length distribution rather than arbitrary cutoffs. The fine-grained view shows three real
clusters and how each rolls up into the assigned bucket.

In [0]:
%sql
SELECT
  CASE
    WHEN duration_seconds < 60 THEN 'a: <1m'
    WHEN duration_seconds < 180 THEN 'b: 1-3m'
    WHEN duration_seconds < 600 THEN 'c: 3-10m'
    WHEN duration_seconds < 1800 THEN 'd: 10-30m'
    WHEN duration_seconds < 3600 THEN 'e: 30-60m'
    WHEN duration_seconds < 14400 THEN 'f: 1-4h'
    ELSE 'g: 4h+'
  END AS fine_bucket,
  duration_bucket AS assigned_bucket,
  COUNT(*) AS videos
FROM workspace.youtube.silver_videos
GROUP BY fine_bucket, duration_bucket
ORDER BY fine_bucket

fine_bucket,assigned_bucket,videos
a: <1m,short,89
b: 1-3m,short,179
c: 3-10m,medium,78
d: 10-30m,medium,86
e: 30-60m,medium,157
f: 1-4h,long,46
g: 4h+,long,313


### Distribution Across Dimensions

A check that each dimension classifies sensibly across the catalog.

In [0]:
%sql
SELECT duration_bucket, COUNT(*) AS videos
FROM workspace.youtube.silver_videos GROUP BY duration_bucket ORDER BY duration_bucket

duration_bucket,videos
long,359
medium,321
short,268


### Game Format × Broadcast Type

Cross-tabulating two dimensions confirms they capture independent attributes — both cash and
tournament content appear in live and edited forms, so format and broadcast type are not
redundant. (The `other` bucket is examined in the Data Quality section.)

In [0]:
%sql
SELECT game_format, is_live_broadcast, COUNT(*) AS videos
FROM workspace.youtube.silver_videos GROUP BY game_format, is_live_broadcast ORDER BY videos DESC

game_format,is_live_broadcast,videos
other,false,306
tournament,true,301
tournament,false,187
cash,false,96
cash,true,39
other,true,19


### Sample Across Content Types

A stratified sample — short clips, cash-game content, and long broadcasts — confirming each
dimension classifies sensibly on real records.

In [0]:
%sql
(SELECT title, duration_bucket, is_live_broadcast, game_format, stop_location, view_count
 FROM workspace.youtube.silver_videos WHERE duration_bucket = 'short' ORDER BY view_count DESC LIMIT 5)
UNION ALL
(SELECT title, duration_bucket, is_live_broadcast, game_format, stop_location, view_count
 FROM workspace.youtube.silver_videos WHERE game_format = 'cash' ORDER BY view_count DESC LIMIT 5)
UNION ALL
(SELECT title, duration_bucket, is_live_broadcast, game_format, stop_location, view_count
 FROM workspace.youtube.silver_videos WHERE duration_bucket = 'long' ORDER BY view_count DESC LIMIT 5)

title,duration_bucket,is_live_broadcast,game_format,stop_location,view_count
Brutal $839K Showdown! 🤑 #tritonpoker #poker #shorts,short,false,tournament,Cyprus,2389021
Tony G vs Jungleman 🔥 #tritonpoker #poker #shorts,short,false,other,Montenegro,2281458
Going Clubbin' #tritonpoker #poker #shorts,short,false,other,Bahamas,1174506
THREE BOARDS. HALF A MILLION. ONE BEER. #shorts,short,false,other,null,840917
What a BRUTAL RUNOUT! 🫣 #tritonpoker #shorts,short,false,other,Vietnam,566029
Biggest pot in TV poker history? Million Euro Cash Game at Triton Poker Super High Roller Series,medium,false,cash,Montenegro,3142008
"Tom ""Durrrr"" Dwan vs Rui Cao - The SICKEST Cash Game Poker hand of ALL TIME 🤯",medium,false,cash,Madrid,1604221
Triton Poker Super High Roller Jeju 2018 Cash Game - Episode 4,medium,false,cash,Jeju,1200013
"$200,000 Buy-In CASH GAME | Triton Poker London 2023 NLH (Pt.2)",long,true,cash,London,1058786
"$1,000,000 Buy-in NLH Special Cash Game",long,true,cash,London,1010492


## Data Quality Findings

Classifying this catalog surfaced several data-quality issues worth documenting — both because
they shaped the analysis decisions above, and because they reveal something about how Triton
manages its content metadata. Three findings stand out: inconsistent location tags, limited
game-format determinability, and a deliberate set of field exclusions.

### Game Format Is Often Undeterminable from Titles

Game format (cash vs. tournament) is reliably classified only when titles carry explicit
markers — buy-ins, event numbers, "cash game." About a third of videos fall into `other`:
short clips that describe a *moment* rather than a format, plus trailers and promos. This isn't
a classification failure — it reflects genuinely inconsistent title conventions, and the `other`
rate is reported honestly rather than forced into a guess.

In [0]:
%sql
SELECT game_format, COUNT(*) AS videos,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM workspace.youtube.silver_videos
GROUP BY game_format ORDER BY videos DESC

game_format,videos,pct
tournament,488,51.5
other,325,34.3
cash,135,14.2


### Tag Metadata Is Unreliable for Classification

Tags were excluded as an authoritative source. Using the derived `stop_location` as a
reference, a subset of videos carry tags naming a *different* Triton location. The conflict
rate is modest (~3%), but it concentrates in recent flagship content (Montenegro 2026, Bahamas
2025) carrying a stale `jeju` tag — pointing to tag reuse in the current upload workflow rather
than legacy drift. The same caution extends to other tag-derived signals (e.g. player names),
which would require validation before use.

In [0]:
%sql
-- Headline: conflict rate
WITH tagged AS (
  SELECT s.video_id, s.stop_location, LOWER(b.raw_json:snippet.tags::string) AS tags_str
  FROM workspace.youtube.bronze_videos_raw b
  JOIN workspace.youtube.silver_videos s ON b.video_id = s.video_id
  WHERE s.stop_matched = true AND b.raw_json:snippet.tags IS NOT NULL
),
conflicts AS (
  SELECT DISTINCT t.video_id FROM tagged t
  JOIN workspace.youtube.ref_stops r
    ON t.tags_str RLIKE r.alias_pattern AND r.canonical_name <> t.stop_location
)
SELECT
  (SELECT COUNT(*) FROM tagged)    AS videos_checked,
  (SELECT COUNT(*) FROM conflicts) AS conflicting_videos,
  ROUND(100.0 * (SELECT COUNT(*) FROM conflicts) / (SELECT COUNT(*) FROM tagged), 1) AS pct_conflicting

videos_checked,conflicting_videos,pct_conflicting
853,24,2.8


In [0]:
%sql
-- Breakdown: which stops get mis-tagged as what, and when
WITH tagged AS (
  SELECT s.video_id, s.stop_location, s.published_at, LOWER(b.raw_json:snippet.tags::string) AS tags_str
  FROM workspace.youtube.bronze_videos_raw b
  JOIN workspace.youtube.silver_videos s ON b.video_id = s.video_id
  WHERE s.stop_matched = true AND b.raw_json:snippet.tags IS NOT NULL
),
conflicts AS (
  SELECT DISTINCT t.video_id, t.stop_location, t.published_at, r.canonical_name AS conflicting_tag_location
  FROM tagged t
  JOIN workspace.youtube.ref_stops r
    ON t.tags_str RLIKE r.alias_pattern AND r.canonical_name <> t.stop_location
)
SELECT stop_location AS actual_stop, conflicting_tag_location, YEAR(published_at) AS year, COUNT(*) AS videos
FROM conflicts
GROUP BY actual_stop, conflicting_tag_location, YEAR(published_at)
ORDER BY videos DESC

actual_stop,conflicting_tag_location,year,videos
Montenegro,Jeju,2026,15
Bahamas,Jeju,2025,7
Montenegro,Jeju,2019,1
Jeju,Montenegro,2019,1
Jeju,London,2019,1


### Fields Considered and Excluded

Most of the raw payload was deliberately not carried into silver: API metadata (`kind`, `etag`),
channel-constant fields, thumbnails, technical attributes (`definition`, `caption`), near-constant
fields (`categoryId` — 99% one value), and the deprecated `favoriteCount`. Two excluded fields
carry latent value for future work: `tags` (player names, pending the reliability caveat above)
and the `liveStreamingDetails` timestamps (which could yield actual broadcast duration).

### Content Tags Don't Form a Usable Taxonomy

Beyond location, a broader question: can the catalog be grouped *by content type* — separating
gameplay from interviews, documentaries, tributes, and promos — using the videos' tags? The
answer is no. Tag keywords that should signal non-gameplay content ("story," "moments,"
"journey," "documentary") are applied indiscriminately across gameplay highlights, and a
corrupted tag (`tpk&x%`) propagates across much of the recent catalog. There is no reliable
programmatic way to isolate content types from the public tag layer.

**Why this matters:** it means Triton cannot currently measure its own content-type performance —
e.g. "how does our story-driven content perform versus gameplay?" — without manual review (as was
required for the storytelling analysis later in this notebook). A coherent content taxonomy would
turn that question from a manual exercise into a systematic, repeatable measurement — the
foundation for data-informed programming decisions.

## Gold Layer — Engagement Analysis

The gold layer aggregates the classified videos into curated analytics. The headline metric is
**engagement rate = (likes + comments) / views**, expressed as a percentage — a normalized
measure of how strongly viewers respond, far more comparable across videos than raw view counts
(which are biased by a video's age). Reach is tracked separately via total and average views.

**An honest limitation:** this is a *public-data proxy*. YouTube's own core engagement signals —
watch time, audience retention, click-through rate — are available only to the channel owner.
Likes and comments are the strongest public proxy available, but they capture interaction, not
attention. (Note also that since March 2025 YouTube counts Shorts views more loosely than
long-form views, so short-vs-long view counts aren't perfectly comparable.)

### Engagement Is Driven by Duration

Short-form content engages roughly 2.5× higher than medium or long-form, while long-form drives
far more total views. This is the clearest and most robust pattern in the data — the
reach-vs-engagement tradeoff.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.youtube.gold_engagement_by_duration AS
SELECT duration_bucket,
  COUNT(*) AS videos,
  SUM(view_count) AS total_views,
  ROUND(AVG(view_count), 0) AS avg_views,
  ROUND(100.0 * (SUM(like_count) + SUM(comment_count)) / NULLIF(SUM(view_count), 0), 3) AS engagement_rate_pct
FROM workspace.youtube.silver_videos
GROUP BY duration_bucket ORDER BY duration_bucket;
SELECT * FROM workspace.youtube.gold_engagement_by_duration

duration_bucket,videos,total_views,avg_views,engagement_rate_pct
long,359,61724738,171935.0,0.728
medium,321,49995848,155750.0,0.649
short,268,20511515,76536.0,1.616


### Short-Form: A Bigger Bet, But a Dilution Signal

Two views together tell a strategy story. First, content mix over time: Triton's short-form
share grew from a sliver of output to ~44% by 2025 — a clear, deliberate bet on short-form.
Second, engagement rate over time: the short-form premium holds every year, but as Triton
tripled short-form output in 2025, *per-video* short engagement fell (from ~1.8% in 2023–24 to
~1.1% in 2025). Producing far more short-form coincided with each piece engaging less — a
credible diminishing-returns signal, since it runs counter to what video age alone would predict.
(Early years are low-volume; the robust trend is 2022 onward.)

In [0]:
%sql
-- Content mix over time (share of each year's output); stacked bar chart: one row per year+bucket
SELECT
  YEAR(published_at) AS year,
  duration_bucket,
  COUNT(*) AS videos,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY YEAR(published_at)), 1) AS pct
FROM workspace.youtube.silver_videos
WHERE YEAR(published_at) < 2026
GROUP BY YEAR(published_at), duration_bucket
ORDER BY year, duration_bucket

year,duration_bucket,videos,pct
2018,long,16,26.7
2018,medium,32,53.3
2018,short,12,20.0
2019,long,47,43.5
2019,medium,53,49.1
2019,short,8,7.4
2020,medium,13,72.2
2020,short,5,27.8
2021,medium,15,83.3
2021,short,3,16.7


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT YEAR(published_at) AS year, duration_bucket,
  COUNT(*) AS videos,
  ROUND(100.0 * (SUM(like_count) + SUM(comment_count)) / NULLIF(SUM(view_count), 0), 3) AS engagement_rate_pct
FROM workspace.youtube.silver_videos
WHERE YEAR(published_at) < 2026
GROUP BY YEAR(published_at), duration_bucket
ORDER BY year, duration_bucket

year,duration_bucket,videos,engagement_rate_pct
2018,long,16,0.378
2018,medium,32,0.340
2018,short,12,0.576
2019,long,47,0.380
2019,medium,53,0.609
2019,short,8,1.338
2020,medium,13,0.697
2020,short,5,0.967
2021,medium,15,0.742
2021,short,3,1.579


Databricks visualization. Run in Databricks to view.

### Apparent Format and Location Effects Are Content-Mix Artifacts

Engagement also appears to vary by game format and by location — but these differences largely
dissolve when controlling for duration. The cross-cut below shows that within each duration tier,
engagement rates converge across game formats: the apparent format differences were mostly a
reflection of how much short-form content each category contained, not an independent effect.
The same holds for location. Duration is the dominant driver.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.youtube.gold_engagement_cross AS
SELECT duration_bucket, game_format,
  COUNT(*) AS videos,
  ROUND(AVG(view_count), 0) AS avg_views,
  ROUND(100.0 * (SUM(like_count) + SUM(comment_count)) / NULLIF(SUM(view_count), 0), 3) AS engagement_rate_pct
FROM workspace.youtube.silver_videos
GROUP BY duration_bucket, game_format
ORDER BY duration_bucket, game_format;
SELECT * FROM workspace.youtube.gold_engagement_cross

duration_bucket,game_format,videos,avg_views,engagement_rate_pct
long,cash,29,310274.0,0.617
long,other,12,78884.0,0.627
long,tournament,318,162831.0,0.749
medium,cash,97,308518.0,0.566
medium,other,91,126615.0,0.730
medium,tournament,133,64268.0,0.828
short,cash,9,17941.0,0.770
short,other,222,66822.0,1.627
short,tournament,37,149071.0,1.612


Databricks visualization. Run in Databricks to view.

### What Survives the Control: Cash Games Drive Reach

One non-duration effect does survive: cash-game content draws the highest *average views per
video* (~290K vs. ~135K for tournaments) — a reach finding, distinct from engagement rate. Cash
games pull big audiences even though they don't engage at higher rates.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.youtube.gold_engagement_by_game_format AS
SELECT game_format,
  COUNT(*) AS videos,
  SUM(view_count) AS total_views,
  ROUND(AVG(view_count), 0) AS avg_views,
  ROUND(100.0 * (SUM(like_count) + SUM(comment_count)) / NULLIF(SUM(view_count), 0), 3) AS engagement_rate_pct
FROM workspace.youtube.silver_videos
GROUP BY game_format ORDER BY total_views DESC;
SELECT * FROM workspace.youtube.gold_engagement_by_game_format

game_format,videos,total_views,avg_views,engagement_rate_pct
tournament,488,65843495,134925.0,0.831
cash,135,39085651,289523.0,0.579
other,325,27302955,84009.0,1.214


## The Opportunity: Story-Driven Content Is Under-Tested, Not Untested

The analysis establishes that short-form drives engagement, that the catalog is overwhelmingly
gameplay, and that genuine narrative content is rare. But "rare" is not "absent" — and the small
amount that exists offers a directional signal.

### A real precedent: the Danny Tang arc

In late 2024, Triton produced a coordinated narrative around Danny Tang's Player of the Year win:
a ceremony piece (Nov 6) followed by a three-part journey documentary, *Race to the Top*
(Nov 30 – Dec 20) — precisely the character-driven, episodic storytelling a storytelling brand
would be expected to invest in. How did it perform?

In [0]:
%sql
-- The Danny Tang narrative arc vs. the medium-duration baseline
SELECT title, ROUND(duration_seconds/60.0,1) AS duration_min,
  view_count, like_count, comment_count,
  ROUND(100.0*(like_count+comment_count)/NULLIF(view_count,0),2) AS eng_rate_pct,
  published_at
FROM workspace.youtube.silver_videos
WHERE video_id IN ('T1F1z_xuX6Y','F2-p37Ln8Kg','os4MZYkMi6c','k_KvMDCIiPw')
ORDER BY published_at

title,duration_min,view_count,like_count,comment_count,eng_rate_pct,published_at
Danny Tang Wins Ivan Leow Player of the Year Award | Triton Poker Series Season 3 Ceremony,11.4,8336,131,19,1.80,2024-11-06T11:00:52.000Z
Danny Tang’s Rise to the Top | Fueled by Brotherhood | Race to the Top (Episode 1),3.4,7358,110,8,1.60,2024-11-30T10:00:25.000Z
Danny Tang's Journey Under Pressure | Race to the Top (Episode 2),4.5,5428,38,3,0.76,2024-12-06T12:00:18.000Z
Danny Tang’s Journey to Triumph | Race to the Top | Triton Ambassador Story (Episode 3),5.3,7546,82,7,1.18,2024-12-20T10:00:50.000Z


The pattern is consistent across the arc: for medium-duration content, the four pieces engaged
*above* the catalog's medium-video baseline of **~0.65%** (individually 1.80%, 1.60%, 0.76%, and
1.18%), yet each reached only **5,000–8,000 views** — against a ~156,000-view average for medium
content. The comparison below makes the contrast explicit: the arc's engagement rate sits well
above baseline, while its average reach is a small fraction of a typical medium video's.

The honest read is not that narrative content failed, but that it was barely tested — a promising
per-view engagement signal, starved of distribution, and never repeated.

In [0]:
%sql
-- Danny Tang narrative arc vs. medium-duration baseline
SELECT
  'Danny Tang arc (narrative)' AS segment,
  COUNT(*) AS videos,
  ROUND(AVG(view_count), 0) AS avg_views,
  ROUND(100.0*(SUM(like_count)+SUM(comment_count))/NULLIF(SUM(view_count),0),3) AS eng_rate_pct
FROM workspace.youtube.silver_videos
WHERE video_id IN ('T1F1z_xuX6Y','F2-p37Ln8Kg','os4MZYkMi6c','k_KvMDCIiPw')
UNION ALL
SELECT
  'All medium-duration (baseline)' AS segment,
  COUNT(*) AS videos,
  ROUND(AVG(view_count), 0) AS avg_views,
  ROUND(100.0*(SUM(like_count)+SUM(comment_count))/NULLIF(SUM(view_count),0),3) AS eng_rate_pct
FROM workspace.youtube.silver_videos
WHERE duration_bucket = 'medium'

segment,videos,avg_views,eng_rate_pct
Danny Tang arc (narrative),4,7167.0,1.388
All medium-duration (baseline),321,155750.0,0.649


### Honest limits and opportunity for scale

The Danny Tang arc is not a turnkey model: it depended on a singular event — only one player wins
Player of the Year — so a strategy hinging on a unique milestone doesn't scale on its own.

But the *framework* — following a player's journey toward a goal — scales if the goal is
structural rather than singular. Triton ONE, the mid-stakes circuit launched in 2025 to "connect
rising talents with the Triton experience," offers exactly that: rather than one champion's story
told after the fact, Triton could follow multiple rising players across a season — an ensemble
narrative with built-in stakes (who breaks through to the Super High Roller stage?), renewable
every cycle.

### The unsolved problem: reach

Even a scalable format faces the distribution problem this content surfaced — it engaged but
barely circulated. These questions outrun what the data can answer, but are worth posing:

- **What would a global brand do differently to promote it?** Was it cross-promoted during
  livestreams — Triton's highest-reach surface — or released to build anticipation, or simply
  posted?
- **Could an audience outside poker care?** *The Last Dance* and *Drive to Survive* built
  mainstream audiences for niche subjects through character and stakes; reaching beyond the poker
  audience requires distribution channels a YouTube-only approach doesn't exercise.
- **What is the carrot?** Gameplay clips offer instant payoff; narrative asks for investment. What
  hook makes a non-fan press play — rivalry, underdog, money, personality?

These are hypotheses for Triton to test, not conclusions the data settles. But the direction is
grounded: the format engages, it can be made to scale, and the binding constraint is distribution
— a solvable problem for a brand of Triton's reach.

## Conclusions

Returning to the opening question — how well does Triton's content strategy live up to its
"premium poker storytelling" positioning, and what drives engagement?

**What the data shows clearly:** Engagement is driven primarily by content *length*. Short-form
engages ~2.5× higher than long-form, consistently across every year, location, and game format.
Apparent differences by location and game type largely dissolve once duration is controlled for —
they were content-mix artifacts, not independent effects. The one reach effect that survives is
that cash-game content draws the largest per-video audiences. And as Triton tripled short-form
output in 2025, per-video engagement declined — a dilution signal suggesting volume alone is
reaching diminishing returns.

**On the brand question:** the catalog is overwhelmingly gameplay. For a self-described
storytelling brand, genuine narrative content is rare — and the metadata can't even reliably
isolate it. Yet the small narrative sample that exists (the Danny Tang arc) engaged above its
baseline while reaching almost no one. As detailed above, this points to a specific, evidence-
grounded opportunity: story-driven content — scaled through an ensemble format like Triton ONE,
and given the distribution it never received — is an under-tested bet that aligns with both the
engagement data and the brand's stated ambition.

**Honest limitations:** this rests on public engagement proxies (likes + comments per view), not
YouTube's owner-only signals (watch time, retention). It is a single snapshot, so trends are
inferred from a static catalog. Classifications are reliable for length and broadcast type,
best-effort for game format (34% undeterminable) and location (90% matched), and the narrative
findings rest on a small, manually-curated sample. The conclusions are directional and honest
about their limits.